In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error

xgb_oof = np.load("../output/oof/xgb_oof.npy")
xgb_test = np.load("../output/oof/xgb_test.npy")
lgb_oof = np.load("../output/oof/lgb_oof.npy")
lgb_test = np.load("../output/oof/lgb_test.npy")
cat_oof = np.load("../output/oof/cat_oof.npy")
cat_test = np.load("../output/oof/cat_test.npy")

y = pd.read_csv("../data/processed/y_train.csv")
y = y.iloc[:, 0].values if y.shape[1] == 1 else y.values.ravel()

oof_matrix = np.vstack([xgb_oof, lgb_oof, cat_oof]).T   # shape: (n_samples, n_models)
test_matrix = np.vstack([xgb_test, lgb_test, cat_test]).T
model_names = ["xgboost", "lightgbm", "catboost"]

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

print("OOF matrix shape:", oof_matrix.shape)
print("Test matrix shape:", test_matrix.shape)

OOF matrix shape: (1458, 3)
Test matrix shape: (1459, 3)


In [2]:
n_models = oof_matrix.shape[1]

def weighted_rmse(weights, oof_matrix, y):
    blend = oof_matrix @ weights
    return rmse(y, blend)

# Start with equal weights
init_weights = np.array([1 / n_models] * n_models)

# Constraint: weights must sum to 1
constraints = ({"type": "eq", "fun": lambda w: np.sum(w) - 1})
# Constraint: each weight between 0 and 1
bounds = [(0, 1)] * n_models

result = minimize(
    weighted_rmse,
    init_weights,
    args=(oof_matrix, y),
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
)

best_weights = result.x
print("Optimal weights:")
for name, w in zip(model_names, best_weights):
    print(f"  {name}: {w:.4f}")

ensemble_oof_rmse = weighted_rmse(best_weights, oof_matrix, y)
print(f"Ensemble OOF RMSE: {ensemble_oof_rmse:.5f}")

Optimal weights:
  xgboost: 0.2981
  lightgbm: 0.0083
  catboost: 0.6936
Ensemble OOF RMSE: 0.11133


In [3]:
single_scores = {name: rmse(y, oof_matrix[:, i]) for i, name in enumerate(model_names)}

comparison = pd.DataFrame({
    "model": list(single_scores.keys()) + ["ensemble"],
    "oof_rmse": list(single_scores.values()) + [ensemble_oof_rmse],
})
comparison = comparison.sort_values("oof_rmse").reset_index(drop=True)
print(comparison)

best_single = min(single_scores.values())
improvement = best_single - ensemble_oof_rmse
improvement_pct = improvement / best_single * 100
print(f"Improvement over best single model: {improvement:.5f} ({improvement_pct:.2f}%)")

import os
os.makedirs("../output/oof", exist_ok=True)
comparison.to_csv("../output/oof/ensemble_comparison.csv", index=False)

      model  oof_rmse
0  ensemble  0.111325
1  catboost  0.112144
2   xgboost  0.115580
3  lightgbm  0.120492
Improvement over best single model: 0.00082 (0.73%)


In [4]:
test_blend_log = test_matrix @ best_weights

# Target was log1p-transformed earlier, so reverse it with expm1
test_blend_price = np.expm1(test_blend_log)

# Get test Ids in the correct order (from original test.csv)
test_ids = pd.read_csv("../data/test.csv")["Id"].values

submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": test_blend_price
})

import os
os.makedirs("../output", exist_ok=True)
submission.to_csv("../output/submission.csv", index=False)
submission.head()

,Id,SalePrice
0,1461,126497.645723
1,1462,161657.078422
2,1463,184107.387115
3,1464,190816.145395
4,1465,184019.958118


In [5]:
sample_sub = pd.read_csv("../data/sample_submission.csv")

checks = {
    "Row count is 1459": len(submission) == 1459,
    "All prices are non-negative": (submission["SalePrice"] >= 0).all(),
    "Id order matches sample_submission": (submission["Id"].values == sample_sub["Id"].values).all(),
}

for check_name, passed in checks.items():
    status = "PASSED" if passed else "FAILED"
    print(f"{check_name}: {status}")

assert all(checks.values()), "Submission file has issues, please check!"

Row count is 1459: PASSED
All prices are non-negative: PASSED
Id order matches sample_submission: PASSED
